In [1]:
import sys, os
%load_ext ElasticNotebook
from elastic.core.common.pandas import compare_df, convert_col
import pickle

Enabled rmm statistics


In [2]:
%load_ext cudf.pandas

In [3]:
%LoadCheckpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/rewritten/o4_mini_high_small/checkpoints/post_cell_9_try_0.pickle

trying: ['BENCHMARKS_TO_PATHS']
me:  0
trying: ['filename']
me:  1
trying: ['benchmark_name']
me:  1
trying: ['factor']
me:  1
trying: ['total_movie_with_gross']
me:  18
trying: ['spiel']
me:  13
trying: ['num_movies_directed']
me:  13
trying: ['e_gross_under_budget']
me:  15
trying: ['e_movies']
me:  15
trying: ['orig_output']
me:  16
trying: ['ratio']
me:  17
trying: ['valid']
me:  17
trying: ['average_gross']
me:  18
trying: ['movie_grossed_over_average']
me:  18
trying: ['ninety_min_num_movies']
me:  9
trying: ['np']
me:  0
trying: ['m']
me:  5
trying: ['total_num_movies']
me:  9
trying: ['pd']
me:  0
trying: ['two_hour_movies']
me:  11
trying: ['Path']
me:  0
Declaring variable BENCHMARKS_TO_PATHS
Declaring variable np
Declaring variable pd
Declaring variable Path
Declaring variable filename
Declaring variable benchmark_name
Declaring variable factor
Declaring variable m
Declaring variable ninety_min_num_movies
Declaring variable total_num_movies
Declaring variable two_hour_movies

In [4]:
%%RecordEvent
%%time
### cell 10 ###

# Optimized GPU-only pipeline
imdb = m.imdb_score
ngross = m.gross
nbudget = m.budget

# build a single boolean mask for (imdb>6) and non‐null gross/budget
good = (imdb > 6) & ngross.notnull() & nbudget.notnull()

# false positives = gross<budget within the good set
# rate = count(false positives) / count(good)
false_positive_rate = ((ngross < nbudget) & good).sum() / good.sum()

false_positive_rate

CPU times: user 43.5 ms, sys: 1.84 ms, total: 45.4 ms
Wall time: 42.9 ms


np.float64(0.43015521064301554)

In [5]:
%Checkpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/rewritten/o4_mini_high_small/checkpoints/post_cell_10_try_2.pickle

migration speed (bps): 819764925.7617339
---------------------------
variables to migrate:
good 1023594
e_gross_under_budget 28
e_movies 28066
imdb 5113568
ratio 32
valid 14781808
orig_output 24
ngross 5113568
np 72
m 18714658
average_gross 32
total_movie_with_gross 32
movie_grossed_over_average 32
false_positive_rate 32
total_num_movies 28
two_hour_movies 28
ninety_min_num_movies 28
pd 72
BENCHMARKS_TO_PATHS 2272
num_movies_directed 28
Path 904
spiel 28
benchmark_name 53
factor 28
filename 88
nbudget 5113568
---------------------------
variables to recompute:
[]
---------------------------
cells to recompute:
[]
Checkpoint saved to: /scratch/jieq/pandax/ds_notebooks/imdb/src/rewritten/o4_mini_high_small/checkpoints/post_cell_10_try_2.pickle


In [6]:
%PrintCellInfo opt_cell_exec_info

======= Cell 0 =======
Input variables ['BENCHMARKS_TO_PATHS', 'Path']
Active variables ['m']
Intermediate variables ['benchmark_name', 'filename', 'factor']
Future variables []
Modified dataframes
======= Cell 1 =======
Input variables ['m']
Active variables []
Intermediate variables []
Future variables []
Modified dataframes
======= Cell 2 =======
Input variables ['m']
Active variables ['m']
Intermediate variables []
Future variables []
Modified dataframes
  m
    Input columns: set()
    Changed columns: {'movie_facebook_likes', 'budget', 'num_voted_users', 'num_critic_for_reviews', 'imdb_score', 'num_user_for_reviews', 'director_name', 'duration', 'title_year', 'content_rating', 'gross', 'genres'}
    Created columns: set()
    Deleted columns: {'facenumber_in_poster', 'movie_title', 'actor_3_facebook_likes', 'language', 'actor_2_facebook_likes', 'actor_2_name', 'plot_keywords', 'actor_3_name', 'movie_imdb_link', 'cast_total_facebook_likes', 'aspect_ratio', 'color', 'country', 'act

In [7]:

with open("/scratch/jieq/pandax/ds_notebooks/imdb/src/opt_cell_exec_info_10_try_2.pkl", "wb") as f:
    pickle.dump(opt_cell_exec_info[10], f)


In [8]:
opt_output = Out.get(4)

In [9]:
movies_with_scores_opt = movies_with_scores
%LoadCheckpoint /scratch/jieq/pandax/ds_notebooks/imdb/src/small_bench/checkpoints/post_cell_10.pickle
assert compare_df(movies_with_scores_opt, movies_with_scores)

import numpy as np
if os.getenv("USE_GPU") == "True":
    import cudf
from elastic.core.common.pandas import is_type_styler
is_orig_output_pd = isinstance(orig_output, (pd.Series, pd.DataFrame, pd.Index))
is_opt_output_pd = isinstance(opt_output, (pd.Series, pd.DataFrame, pd.Index))
if os.getenv("USE_GPU") == "True":
    is_orig_output_array = isinstance(orig_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
    is_opt_output_array = isinstance(opt_output, (cudf.pandas._wrappers.numpy.ndarray, np.ndarray))
else:
    is_orig_output_array = isinstance(orig_output, np.ndarray)
    is_opt_output_array = isinstance(opt_output, np.ndarray)

is_orig_output_styler = is_type_styler(type(orig_output))
is_opt_output_styler = is_type_styler(type(opt_output))
if is_orig_output_styler and is_opt_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_orig_output_styler:
    assert orig_output.to_html() == opt_output.to_html()
elif is_opt_output_styler:
    assert opt_output.to_html() == orig_output

if is_orig_output_pd and is_opt_output_pd:
    assert orig_output.equals(opt_output)
# TODO: this is a hack.
elif ((is_orig_output_pd or is_opt_output_pd) and (is_orig_output_array or is_opt_output_array)) or (is_orig_output_array and is_opt_output_array):
    assert list(orig_output) == list(opt_output)
else:
    assert orig_output == opt_output


NameError: name 'movies_with_scores' is not defined